In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
from dotenv import load_dotenv
import requests
import os

/home/kamadahanam/Documents/ALL_WORKS/Programming/LangChain/.venv/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [2]:
load_dotenv('../.env')

True

In [3]:
from langchain_community.tools import DuckDuckGoSearchRun

search_tool = DuckDuckGoSearchRun()

In [4]:
WEATHERSTACK_KEY=os.getenv('WEATHERSTACK_API_KEY')

In [5]:
@tool
def get_weather_data(city: str) -> str:
  """
  This function fetches the current weather data for a given city
  """
  
  url = f'http://api.weatherstack.com/current?access_key={WEATHERSTACK_KEY}&query={city}'

  response = requests.get(url)

  return response.json()

In [6]:
llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

In [7]:
from langchain_classic.agents import create_react_agent, AgentExecutor
from langchain_classic import hub

In [8]:
# Step 2: Pull the ReAct prompt from LangChain Hub
prompt = hub.pull("hwchase17/react")  # pulls the standard ReAct agent prompt


In [9]:
# Step 3: Create the ReAct agent manually with the pulled prompt
agent = create_react_agent(
    llm=llm,
    tools=[search_tool, get_weather_data],
    prompt=prompt
)

In [10]:
# Step 4: Wrap it with AgentExecutor
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool, get_weather_data],
    verbose=True
)

In [13]:
# Step 5: Invoke
response = agent_executor.invoke({"input": "Find the capital of Czech Republic and the top news realted to it, then find it's capital's current weather condition"})
print(response)



> Entering new AgentExecutor chain...
Action: duckduckgo_search
Action Input: capital of Czech RepublicPrague is the capital of the Czech Republic and, as such, is the regular seat of its central authorities. Since 24 November 1990, it is de facto again a statutory city, but has a specific status of the municipality and the region at the same time. Prague, city, capital of the Czech Republic . Lying at the heart of Europe, it is one of the continent’s finest cities and the major Czech economic and cultural centre. The city has a rich architectural heritage that reflects both the uncertain currents of history in Bohemia and an urban life extending back more than 1,000 years. The physical attractions and landmarks of Prague are many. Among the finest is the Charles Bridge (Karlův most), which stands astride the Vltava River. The winding course of the Vltava, with its succession of bridges and changing vistas, contrasts with the ever-present backdrop of the great castle of Hradčany (Pra

In [14]:
response['output']

'The capital of the Czech Republic is Prague. Top news related to Prague includes: "Prague police are investigating a false school shooting report made by a teacher; students are safe, and the teacher faces possible prison time." and "Ice warning issued across parts of Moravia, Prague builds fewest offices since 1993, and more weekend headlines." The current weather in Prague is partly cloudy with a temperature of 1°C, feeling like -1°C, wind speed of 6 km/h from WSW, pressure of 1002 hPa, humidity of 75%, and cloudcover of 75%.'